In [11]:
import pandas as pd
import numpy as np
import re
import pickle

# Traditional Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [12]:
print("Loading data...")
# Using relative path so it works for the whole team on GitHub
df = pd.read_csv('spam.csv', encoding='latin-1')

# The dataset has empty columns at the end, we only need 'v1' (label) and 'v2' (message)
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Convert string labels to numbers: 'ham' = 0, 'spam' = 1
encoder = LabelEncoder()
df['label_num'] = encoder.fit_transform(df['label'])

Loading data...


In [13]:
def clean_text(text):
    text = text.lower()
    # Replace any web address with the word "URLTAG"
    text = re.sub(r'http\S+|www\.\S+|\S+\.com', 'urltag', text)
    # Then remove punctuation (but keep spaces and letters)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

print("Cleaning text...")
df['clean_message'] = df['message'].apply(clean_text)

Cleaning text...


In [14]:
print("Vectorizing text with TF-IDF...")
# max_features limits the vocabulary to the top 10,000 most frequent words (same as CNN)
# (1, 2) means look at single words AND pairs of words next to each other
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Fit the vectorizer on the text and transform it into a numerical matrix
X = vectorizer.fit_transform(df['clean_message'])
y = df['label_num']

Vectorizing text with TF-IDF...


In [15]:
print("Building Random Forest Model...")
# n_estimators=100 means the forest will have 100 decision trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# n_jobs=-1 tells the computer to use all CPU cores to make it train faster


Building Random Forest Model...


In [16]:
print("Training...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

# Check accuracy
print("\nEvaluating model...")
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Optional but highly recommended for the doctor discussion
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Training...

Evaluating model...
Test Accuracy: 97.13%

--- Classification Report ---
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       965
        Spam       1.00      0.79      0.88       150

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [17]:
print("Saving model and vectorizer...")
# Save the Random Forest Model
with open('rf_spam_model.pkl', 'wb') as file:
    pickle.dump(rf_model, file)

# SAVE THE VECTORIZER! The GUI student needs this to convert user input into TF-IDF numbers
with open('rf_vectorizer.pickle', 'wb') as file:
    pickle.dump(vectorizer, file, protocol=pickle.HIGHEST_PROTOCOL)

print("Training complete! Files saved: 'rf_spam_model.pkl' and 'rf_vectorizer.pickle'")


Saving model and vectorizer...
Training complete! Files saved: 'rf_spam_model.pkl' and 'rf_vectorizer.pickle'


In [18]:
def predict_spam_rf(input_text):
    """Function to be called by the GUI to get a prediction."""
    # 1. Clean the input text exactly like we did during training
    cleaned_text = clean_text(input_text)
    
    # 2. Convert to TF-IDF matrix (Notice we use .transform(), NOT .fit_transform()!)
    tfidf_matrix = vectorizer.transform([cleaned_text])
    
    # 3. Predict
    prediction_num = rf_model.predict(tfidf_matrix)[0]
    
    # 4. Return result based on the number
    if prediction_num == 1:
        return "Spam"
    else:
        return "Ham"

In [20]:
test_msg = "to reset your password, click the link below and follow the instructions."
print(f"\nTest Prediction for: '{test_msg}'")
print(f"Result: {predict_spam_rf(test_msg)}")


Test Prediction for: 'to reset your password, click the link below and follow the instructions.'
Result: Ham
